In [1]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import pickle
import math

# ==========================================
# 1. 数据加载与超参数提取
# ==========================================

# 使用 torch.load 直接加载 .pt 文件
# 建议加上 map_location='cpu'，防止保存时使用了 GPU，而读取时在 CPU 环境下报错
# weights_only=True 是 PyTorch 2.0+ 的安全规范，推荐开启
Data_AllOrg = torch.load("./AllData/Pichia_2Target.pt", map_location='cpu', weights_only=True)

# 选取前500条数据验证训练过程   
# AA_tr = Data_AllOrg['AA_tr'][:500]
# Cds_tr = Data_AllOrg['Cds_tr'][:500]

AA_tr = Data_AllOrg['AA_tr']
Cds_tr = Data_AllOrg['Cds_tr']

# 读取超参数配置 (确保文件路径正确)
Settings = pd.read_csv('./BO_forHyperParameter/Arch1/Round3.csv').iloc[:, 1:]
Setting_no = 1

Max_length = 1110
learning_rate = 0.0005
batch_size = 16
epochs = 100
aa_vocab_size = 25
dna_vocab_size = 67

# --- ADD THIS TO CHECK/FIX THE TRUE VOCAB SIZE ---
# Find the maximum integer index in your tensors and add 1 
# (because indices start at 0, so max index 66 means size 67)
actual_aa_vocab_size = AA_tr.max().item() + 1
actual_dna_vocab_size = Cds_tr.max().item() + 1

print(f"Config aa_vocab: {aa_vocab_size}, Actual required: {actual_aa_vocab_size}")
print(f"Config dna_vocab: {dna_vocab_size}, Actual required: {actual_dna_vocab_size}")

# Override the hardcoded settings with the safe sizes
aa_vocab_size = max(aa_vocab_size, actual_aa_vocab_size)
dna_vocab_size = max(dna_vocab_size, actual_dna_vocab_size)
# -------------------------------------------------





hidden_size_enc = int(Settings['Enc hidden size'][Setting_no])
hidden_size_enc_aa = int(Settings['Enc hidden size'][Setting_no])
embedding_size_enc = int(Settings['Enc Embedding size'][Setting_no])
embedding_size_dec = int(Settings['Dec Embedding size'][Setting_no])
Dense_layer_size = int(Settings['Dense Layer size'][Setting_no])
Dense_layer_size_aa = int(Settings['Dense Layer size aa'][Setting_no])
drop_rate = Settings['Drop rate'][Setting_no]
drop_rate_aa = Settings['Drop rate aa'][Setting_no]

# ==========================================
# 2. PyTorch Dataset 与 Train/Val 切分
# ==========================================

class Seq2SeqDataset(Dataset):
    def __init__(self, aa_data, cds_data, max_len):
        # 修复警告：如果输入已经是 Tensor，使用 clone().detach() 更安全、更高效
        self.aa = aa_data.clone().detach().long()
        self.cds = cds_data.clone().detach().long()
        self.max_len = max_len

    def __len__(self):
        return len(self.aa)

    def __getitem__(self, idx):
        # 截取固定长度，防止个别数据超长
        aa = self.aa[idx][:self.max_len]
        cds = self.cds[idx][:self.max_len]
        
        # 修复 RuntimeError：统一所有序列的长度为 max_len - 1 (即 999)
        # Encoder 端：输入氨基酸序列 (忽略开头的 SOS 符)
        enc_input = aa[1:]          
        
        # Decoder 端 CDS：输入去尾 (不要 EOS)，目标去头 (不要 SOS)
        dec_input_cds = cds[:-1]    
        target_cds = cds[1:]        
        
        # Decoder 端 AA：输入去尾，目标去头
        dec_input_aa = aa[:-1]      
        target_aa = aa[1:]          
        
        return enc_input, dec_input_cds, dec_input_aa, target_cds, target_aa



# 初始化全量数据集
full_dataset = Seq2SeqDataset(AA_tr, Cds_tr, Max_length)

# 划分训练集和验证集 (80% 训练, 20% 验证)
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# 构建 DataLoader
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# ==========================================
# 3. 构建 PyTorch 模型
# ==========================================
class MultiTaskSeq2Seq(nn.Module):
    def __init__(self):
        super(MultiTaskSeq2Seq, self).__init__()
        
        # 编码器
        # padding_idx=0 模拟 Keras 中的 mask_zero=True
        self.enc_emb = nn.Embedding(aa_vocab_size, embedding_size_enc, padding_idx=0)
        self.encoder = nn.GRU(embedding_size_enc, hidden_size_enc, batch_first=True, bidirectional=True)
        
        # 密码子 (CDS) 解码器
        self.dec_emb_cds = nn.Embedding(dna_vocab_size, embedding_size_dec, padding_idx=0)
        self.decoder_cds = nn.GRU(embedding_size_dec, 2 * hidden_size_enc, batch_first=True)
        
        # 氨基酸 (AA) 解码器
        self.decoder_aa = nn.GRU(embedding_size_enc, 2 * hidden_size_enc_aa, batch_first=True)
        
        # 中间投影与输出层
        # 输入维度: GRU隐藏层维度 (2*hidden) + Attention上下文维度 (2*hidden) = 4*hidden
        self.inter_dense_cds = nn.Linear(4 * hidden_size_enc, Dense_layer_size)
        self.inter_dense_aa = nn.Linear(2 * hidden_size_enc_aa + 2 * hidden_size_enc, Dense_layer_size_aa)
        
        self.dropout_cds = nn.Dropout(drop_rate)
        self.dropout_aa = nn.Dropout(drop_rate_aa)
        
        self.out_cds = nn.Linear(Dense_layer_size, dna_vocab_size)
        self.out_aa = nn.Linear(Dense_layer_size_aa, aa_vocab_size)

    def dot_product_attention(self, query, value):
        
        # 获取特征维度用于缩放 (防止内积数值过大导致 Softmax 梯度消失/爆炸)
        d_k = query.size(-1) 
        
        # 计算注意力分数并除以 sqrt(d_k)
        scores = torch.bmm(query, value.transpose(1, 2)) / math.sqrt(d_k)
        attn_weights = F.softmax(scores, dim=-1)
        context = torch.bmm(attn_weights, value)
        return context

        

    def forward(self, enc_in, dec_in_cds, dec_in_aa):
        # 1. Encoder 阶段
        enc_emb_out = self.enc_emb(enc_in)
        enc_seq, enc_hidden = self.encoder(enc_emb_out)
        
        # 将双向 GRU 的状态拼接，形状从 (2, batch, hidden) 变为 (1, batch, 2*hidden) 供给解码器
        enc_hidden_concat = torch.cat([enc_hidden[0], enc_hidden[1]], dim=-1).unsqueeze(0)
        
        # 2. Decoder 阶段 (CDS target)
        dec_emb_cds_out = self.dec_emb_cds(dec_in_cds)
        dec_seq_cds, _ = self.decoder_cds(dec_emb_cds_out, enc_hidden_concat)
        attn_out_cds = self.dot_product_attention(dec_seq_cds, enc_seq)
        concat_cds = torch.cat([dec_seq_cds, attn_out_cds], dim=-1)
        
        inter_cds = torch.tanh(self.inter_dense_cds(concat_cds))
        inter_cds = self.dropout_cds(inter_cds)
        logits_cds = self.out_cds(inter_cds) # 输出前不需要 Softmax，PyTorch 的 CrossEntropyLoss 内置了
        
        # 3. Decoder 阶段 (AA target)
        dec_emb_aa_out = self.enc_emb(dec_in_aa) # 共享 Encoder 的 AA Embedding
        dec_seq_aa, _ = self.decoder_aa(dec_emb_aa_out, enc_hidden_concat)
        attn_out_aa = self.dot_product_attention(dec_seq_aa, enc_seq)
        concat_aa = torch.cat([dec_seq_aa, attn_out_aa], dim=-1)
        
        inter_aa = torch.tanh(self.inter_dense_aa(concat_aa))
        inter_aa = self.dropout_aa(inter_aa)
        logits_aa = self.out_aa(inter_aa)
        
        return logits_cds, logits_aa

    

# ==========================================
# 4. 训练与评估逻辑
# ==========================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiTaskSeq2Seq().to(device)

# 建议初始学习率可以稍微调小一点，比如 5e-4 或 1e-3
optimizer = optim.Adam(model.parameters(), lr=learning_rate, eps=1e-6)

# 添加学习率调度器：当 val_loss 连续 3 个 epoch 不下降时，学习率乘以 0.5
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True, min_lr=1e-6
)

# ignore_index=0 告诉损失函数忽略 Padding 的部分，这是非常正确的做法
criterion = nn.CrossEntropyLoss(ignore_index=0)



# Early Stopping 参数初始化
best_val_loss = float('inf')
patience_counter = 0
patience = 10
checkpoint_path = "./PichiaData/2Target_AllData/Arch1-0404.weights.pt"
os.makedirs(os.path.dirname(checkpoint_path), exist_ok=True)

print(f"开始训练，使用设备: {device}")


for epoch in range(epochs):
    # --- 训练阶段 ---
    model.train()
    train_loss = 0.0
    
    for batch_data in train_loader:
        enc_input, dec_input, dec_input_aa, target_cds, target_aa = [d.to(device) for d in batch_data]
        
        optimizer.zero_grad()
        logits_cds, logits_aa = model(enc_input, dec_input, dec_input_aa)
        
        loss_cds = criterion(logits_cds.reshape(-1, dna_vocab_size), target_cds.reshape(-1))
        loss_aa = criterion(logits_aa.reshape(-1, aa_vocab_size), target_aa.reshape(-1))
        
        # 优化 Loss 计算：赋予权重 (可根据实际情况调整，例如赋予难收敛的任务更高权重)
        weight_cds = 0.9
        weight_aa = 0.1
        loss = weight_cds * loss_cds + weight_aa * loss_aa
        
       
        # ==========================================
        # 新增：NaN 拦截器。如果 Loss 爆炸成 NaN，直接丢弃这个 Batch，保护模型！
        # ==========================================
        if torch.isnan(loss) or torch.isinf(loss):
            print("  [!] 警告: 检测到 NaN/Inf 损失，跳过当前 Batch 更新！")
            continue 

        loss.backward()
        
        # ==========================================
        # 修改：收紧梯度裁剪阈值，从 5.0 降到 1.0
        # ==========================================
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        train_loss += loss.item()

    # --- 验证阶段 ---
    model.eval()
    val_loss = 0.0
    correct_cds, total_cds = 0, 0
    correct_aa, total_aa = 0, 0
    
    with torch.no_grad():
        for batch_data in val_loader:
            enc_input, dec_input, dec_input_aa, target_cds, target_aa = [d.to(device) for d in batch_data]
            
            logits_cds, logits_aa = model(enc_input, dec_input, dec_input_aa)
            
            loss_cds = criterion(logits_cds.reshape(-1, dna_vocab_size), target_cds.reshape(-1))
            loss_aa = criterion(logits_aa.reshape(-1, aa_vocab_size), target_aa.reshape(-1))
            val_loss += (loss_cds.item() + loss_aa.item())
            
            # --- 计算准确率 (%) ---
            # 1. 获取预测类别 (argmax)
            preds_cds = torch.argmax(logits_cds, dim=-1)
            preds_aa = torch.argmax(logits_aa, dim=-1)
            
            # 2. 过滤掉 Padding (0) 不计算在准确率内
            mask_cds = (target_cds != 0)
            mask_aa = (target_aa != 0)
            
            # 3. 统计正确个数和有效总数
            correct_cds += (preds_cds[mask_cds] == target_cds[mask_cds]).sum().item()
            total_cds += mask_cds.sum().item()
            
            correct_aa += (preds_aa[mask_aa] == target_aa[mask_aa]).sum().item()
            total_aa += mask_aa.sum().item()

    avg_train_loss = train_loss / len(train_loader)
    avg_val_loss = val_loss / len(val_loader)
    val_acc_cds = (correct_cds / total_cds) * 100 if total_cds > 0 else 0
    val_acc_aa = (correct_aa / total_aa) * 100 if total_aa > 0 else 0

      
    # 获取当前学习率打印出来观察
    current_lr = optimizer.param_groups[0]['lr']
    
    print(f"Epoch [{epoch+1}/{epochs}] | LR: {current_lr:.6f} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"   -> Val Accuracy: CDS = {val_acc_cds:.2f}% | AA = {val_acc_aa:.2f}%")

    # 步进学习率调度器 (根据当前的验证集 Loss 决定是否降低学习率)
    scheduler.step(avg_val_loss)

    # --- Early Stopping 与 Checkpoint 逻辑 ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), checkpoint_path)
        print(f"   [*] 验证集损失下降，已保存模型权重至 {checkpoint_path}")
    else:
        patience_counter += 1
        # 注意：这里的 patience 建议设置得比 scheduler 的 patience 大（例如 scheduler=3, early_stop=10）
        # 这样网络有机会在降学习率后继续尝试收敛
        if patience_counter >= patience:
            print(f"触发早停 (Early Stopping)！在 Epoch {epoch+1} 停止训练。")
            break
    

Config aa_vocab: 25, Actual required: 25
Config dna_vocab: 67, Actual required: 67


/home/owen/anaconda3/envs/esm3/lib/python3.13/site-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


开始训练，使用设备: cuda
Epoch [1/100] | LR: 0.000500 | Train Loss: 2.0336 | Val Loss: 1.0579
   -> Val Accuracy: CDS = 49.39% | AA = 99.95%
   [*] 验证集损失下降，已保存模型权重至 ./PichiaData/2Target_AllData/Arch1-0404.weights.pt
Epoch [2/100] | LR: 0.000500 | Train Loss: 0.9450 | Val Loss: 1.0504
   -> Val Accuracy: CDS = 50.06% | AA = 99.93%
   [*] 验证集损失下降，已保存模型权重至 ./PichiaData/2Target_AllData/Arch1-0404.weights.pt
Epoch [3/100] | LR: 0.000500 | Train Loss: 0.9314 | Val Loss: 1.0315
   -> Val Accuracy: CDS = 51.29% | AA = 99.97%
   [*] 验证集损失下降，已保存模型权重至 ./PichiaData/2Target_AllData/Arch1-0404.weights.pt
Epoch [4/100] | LR: 0.000500 | Train Loss: 0.9019 | Val Loss: 1.0005
   -> Val Accuracy: CDS = 53.71% | AA = 99.97%
   [*] 验证集损失下降，已保存模型权重至 ./PichiaData/2Target_AllData/Arch1-0404.weights.pt
Epoch [5/100] | LR: 0.000500 | Train Loss: 0.8459 | Val Loss: 0.9548
   -> Val Accuracy: CDS = 57.19% | AA = 99.97%
   [*] 验证集损失下降，已保存模型权重至 ./PichiaData/2Target_AllData/Arch1-0404.weights.pt
Epoch [6/100] | LR: 0.000500 

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

# ==========================================
# 1. 全局特殊 Token 
# ==========================================
PAD_IDX = 0
AA_UNK_IDX = 21
AA_EOS_IDX = 23
AA_SOS_IDX = 24
CDS_UNK_IDX = 0
CDS_SOS_IDX = 65
CDS_EOS_IDX = 66

def translate_cds_to_aa(cds_seq):
    """
    
    将核苷酸序列 (CDS) 翻译为氨基酸序列 (AA)
    """
    # ==========================================
    # 1. 验证核苷酸长度是否为3的整数倍
    # ==========================================
    if len(cds_seq) % 3 != 0:
        raise ValueError(f"翻译失败：核苷酸序列长度为 {len(cds_seq)}，不是 3 的整数倍！")

    # ==========================================
    # 2. 准备字典 (使用代码中已有的 dic_AA_codon)
    # ==========================================
    dic_AA_codon = {
        'A': ['GCT', 'GCC', 'GCA', 'GCG'], 'C': ['TGT', 'TGC'],
        'D': ['GAT', 'GAC'], 'E': ['GAA', 'GAG'],
        'F': ['TTT', 'TTC'], 'G': ['GGT', 'GGA', 'GGC', 'GGG'],
        'H': ['CAT', 'CAC'], 'I': ['ATT', 'ATC', 'ATA'],
        'K': ['AAA', 'AAG'], 'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
        'M': ['ATG'], 'N': ['AAT', 'AAC'],
        'P': ['CCT', 'CCC', 'CCA', 'CCG'], 'Q': ['CAA', 'CAG'],
        'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
        'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
        'T': ['ACT', 'ACC', 'ACA', 'ACG'], 'V': ['GTT', 'GTC', 'GTA', 'GTG'],
        'W': ['TGG'], 'Y': ['TAT', 'TAC'],
        '*': ['TAA', 'TAG', 'TGA']
    }
    
    # 构建反向映射字典：密码子 -> 氨基酸 (Codon -> AA)
    codon_to_aa = {}
    for aa, codons in dic_AA_codon.items():
        for codon in codons:
            codon_to_aa[codon] = aa

    # ==========================================
    # 3. 切分核苷酸并进行翻译
    # ==========================================
    aa_seq = ""
    for i in range(0, len(cds_seq), 3):
        codon = cds_seq[i:i+3]
        # 如果遇到未知的密码子（比如含有 N 碱基），返回占位符 'X' 防止程序崩溃
        aa = codon_to_aa.get(codon, 'X')
        aa_seq += aa
        
    return aa_seq
def build_vocabularies():
    # 1. 氨基酸字典
    aa_vocab = {
        'A': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5, 'G': 6, 'H': 7, 'I': 8, 'K': 9, 'L': 10,
        'M': 11, 'N': 12, 'P': 13, 'Q': 14, 'R': 15, 'S': 16, 'T': 17, 'V': 18, 'W': 19,
        'Y': 20, 'X': 21, 'Z': 21, 'B': 21, 'U': 21, 'O': 21, '*': 22
    }
    
    # 2. 密码子字典
    dic_AA_codon = {
        'A': ['GCT', 'GCC', 'GCA', 'GCG'], 'C': ['TGT', 'TGC'],
        'D': ['GAT', 'GAC'], 'E': ['GAA', 'GAG'],
        'F': ['TTT', 'TTC'], 'G': ['GGT', 'GGA', 'GGC', 'GGG'],
        'H': ['CAT', 'CAC'], 'I': ['ATT', 'ATC', 'ATA'],
        'K': ['AAA', 'AAG'], 'L': ['TTA', 'TTG', 'CTT', 'CTC', 'CTA', 'CTG'],
        'M': ['ATG'], 'N': ['AAT', 'AAC'],
        'P': ['CCT', 'CCC', 'CCA', 'CCG'], 'Q': ['CAA', 'CAG'],
        'R': ['CGT', 'CGC', 'CGA', 'CGG', 'AGA', 'AGG'],
        'S': ['TCT', 'TCC', 'TCA', 'TCG', 'AGT', 'AGC'],
        'T': ['ACT', 'ACC', 'ACA', 'ACG'], 'V': ['GTT', 'GTC', 'GTA', 'GTG'],
        'W': ['TGG'], 'Y': ['TAT', 'TAC'],
        '*': ['TAA', 'TAG', 'TGA']
    }
    
    codon_vocab = {}
    idx = 1
    for aa in dic_AA_codon:
        for codon in dic_AA_codon[aa]:
            codon_vocab[codon] = idx
            idx += 1
            
    idx_to_codon = {v: k for k, v in codon_vocab.items()}
    
    # ==========================================
    # 核心新增：构建 AA_ID -> 对应的多个 Codon_ID 的映射表
    # 这就是你 Keras 代码中 token_AA_codon 扮演的角色
    # ==========================================
    aa_id_to_codon_ids = {}
    for aa, aa_id in aa_vocab.items():
        if aa in dic_AA_codon:
            valid_codons = dic_AA_codon[aa]
            aa_id_to_codon_ids[aa_id] = [codon_vocab[c] for c in valid_codons]
            
    return aa_vocab, idx_to_codon, aa_id_to_codon_ids

# ==========================================
# 2. 模型架构定义 (保持不变)
# ==========================================
class MultiTaskSeq2Seq(nn.Module):
    def __init__(self, aa_vocab_size=25, dna_vocab_size=67, 
                 hidden_size_enc=510, hidden_size_enc_aa=510, 
                 embedding_size_enc=42, embedding_size_dec=224, 
                 Dense_layer_size=125, Dense_layer_size_aa=139, 
                 drop_rate=0.0, drop_rate_aa=0.0):
        super(MultiTaskSeq2Seq, self).__init__()
        self.enc_emb = nn.Embedding(aa_vocab_size, embedding_size_enc, padding_idx=PAD_IDX)
        self.encoder = nn.GRU(embedding_size_enc, hidden_size_enc, batch_first=True, bidirectional=True)
        self.dec_emb_cds = nn.Embedding(dna_vocab_size, embedding_size_dec, padding_idx=PAD_IDX)
        self.decoder_cds = nn.GRU(embedding_size_dec, 2 * hidden_size_enc, batch_first=True)
        self.decoder_aa = nn.GRU(embedding_size_enc, 2 * hidden_size_enc_aa, batch_first=True)
        self.inter_dense_cds = nn.Linear(4 * hidden_size_enc, Dense_layer_size)
        self.inter_dense_aa = nn.Linear(2 * hidden_size_enc_aa + 2 * hidden_size_enc, Dense_layer_size_aa)
        self.dropout_cds = nn.Dropout(drop_rate)
        self.dropout_aa = nn.Dropout(drop_rate_aa)
        self.out_cds = nn.Linear(Dense_layer_size, dna_vocab_size)
        self.out_aa = nn.Linear(Dense_layer_size_aa, aa_vocab_size)

    def dot_product_attention(self, query, value):
        d_k = query.size(-1) 
        scores = torch.bmm(query, value.transpose(1, 2)) / math.sqrt(d_k)
        attn_weights = F.softmax(scores, dim=-1)
        return torch.bmm(attn_weights, value)

    def forward(self, enc_in, dec_in_cds, dec_in_aa): pass 

# ==========================================
# 3. 带规则掩码的步进解码
# ==========================================
def translate_aa_to_cds(model, aa_tensor, aa_id_to_codon_ids, device):
    model.eval()
    with torch.no_grad():
        enc_emb_out = model.enc_emb(aa_tensor)
        enc_seq, enc_hidden = model.encoder(enc_emb_out)
        hidden_state = torch.cat([enc_hidden[0], enc_hidden[1]], dim=-1).unsqueeze(0)
        
        decoder_input = torch.tensor([[CDS_SOS_IDX]], dtype=torch.long).to(device)
        predicted_cds_indices = []
        
        # sequence_len 是当前输入序列的长度 (不包含末尾可能会遇到的 EOS)
        sequence_len = aa_tensor.size(1) 
        
        # counter 对应你 Keras 代码里的计数器，步步对应氨基酸
        for counter in range(sequence_len):
            current_aa_id = aa_tensor[0, counter].item()
            
            # 如果翻译到了输入的结尾(EOS)，则终止
            if current_aa_id == AA_EOS_IDX:
                break
                
            dec_emb_cds_out = model.dec_emb_cds(decoder_input)
            dec_seq_cds, hidden_state = model.decoder_cds(dec_emb_cds_out, hidden_state)
            
            attn_out_cds = model.dot_product_attention(dec_seq_cds, enc_seq)
            concat_cds = torch.cat([dec_seq_cds, attn_out_cds], dim=-1)
            
            inter_cds = torch.tanh(model.inter_dense_cds(concat_cds))
            logits_cds = model.out_cds(inter_cds) 
            
            # 提取最后一步的预测得分 [1, dna_vocab_size]
            current_logits = logits_cds[:, -1, :]
            
            # ==========================================
            # 复刻你的掩码逻辑！(Masking)
            # ==========================================
            # 拿到当前氨基酸对应的所有合法密码子 ID
            valid_codon_ids = aa_id_to_codon_ids.get(current_aa_id, [])
            
            if valid_codon_ids:
                # 创一个全是负无穷的 Mask，屏蔽掉词表中所有的概率
                mask = torch.full_like(current_logits, float('-inf'))
                # 只把合法的密码子位置置为 0 (-inf + 0 = -inf, Logit + 0 = Logit)
                for cid in valid_codon_ids:
                    mask[0, cid] = 0.0
                
                # 给原始得分加上 Mask
                current_logits = current_logits + mask
            
            # 在被规则卡死合法选项里，挑出模型认为打分最高的一个
            next_token = torch.argmax(current_logits, dim=-1).item()
            
            predicted_cds_indices.append(next_token)
            
            # 更新状态进入下一步循环
            decoder_input = torch.tensor([[next_token]], dtype=torch.long).to(device)
            
    return predicted_cds_indices

# ==========================================
# 4. 执行主程序
# ==========================================
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 拿到映射表
    aa_vocab, idx_to_codon, aa_id_to_codon_ids = build_vocabularies()
    
    model = MultiTaskSeq2Seq().to(device)
    checkpoint_path = "./PichiaData/2Target_AllData/Arch1-0404.weights.pt"
    # 加载你的权重，如果本地跑报错把 weights_only=True 删掉
    model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
    
    input_aa_string = "DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL" 
    print(f"\n输入氨基酸序列: {input_aa_string}")
    
    # 构建张量，不用 Padding，不用加 SOS
    aa_indices = [aa_vocab.get(char, AA_UNK_IDX) for char in input_aa_string] + [AA_EOS_IDX]
    aa_tensor = torch.tensor([aa_indices], dtype=torch.long).to(device)
    
    # 丢进去预测 (传入掩码映射表)
    predicted_indices = translate_aa_to_cds(model, aa_tensor, aa_id_to_codon_ids, device)
    
    # 拼接结果
    predicted_cds_string = "".join([idx_to_codon.get(idx, "") for idx in predicted_indices])
            
    print(f"生成的 CDS ID 序列: {predicted_indices}")
    print(f"预测的核苷酸序列: {predicted_cds_string}")
    print(f"翻译的核苷酸序列: {translate_cds_to_aa(predicted_cds_string)}")

if __name__ == "__main__":
    main()


输入氨基酸序列: DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL
生成的 CDS ID 序列: [8, 1, 18, 22, 47, 9, 57, 1, 17, 39, 11, 22, 8, 25, 15, 9, 9, 31, 12, 23, 1, 24, 55, 24, 21, 1, 11, 2, 37, 60, 26, 37, 37, 5, 35, 11, 9, 7, 17, 56, 22, 25, 56, 31, 9, 58, 51, 10, 11, 2, 22, 51, 5, 55, 1, 7, 10, 47, 1, 9, 32, 5, 7, 22, 47, 25, 17, 51, 25, 11, 13, 7, 22, 26, 5, 51, 55, 1, 53, 25, 39, 9, 51, 61, 14, 10, 30, 3, 8, 5, 5, 3, 23, 37, 9, 35, 10, 41, 31, 10, 6, 11, 24, 37, 17, 2

In [4]:
HSA = "GATGCACACAAGAGTGAGGTTGCTCATCGGTTTAAAGATTTGGGAGAAGAAAATTTCAAAGCCTTGGTGTTGATTGCCTTTGCTCAGTATCTTCAGCAGTGTCCATTTGAAGATCATGTAAAATTAGTGAATGAAGTAACTGAATTTGCAAAAACATGTGTTGCTGATGAGTCAGCTGAAAATTGTGACAAATCACTTCATACCCTTTTTGGAGACAAATTATGCACAGTTGCAACTCTTCGTGAAACCTATGGTGAAATGGCTGACTGCTGTGCAAAACAAGAACCTGAGAGAAATGAATGCTTCTTGCAACACAAAGATGACAACCCAAACCTCCCCCGATTGGTGAGACCAGAGGTTGATGTGATGTGCACTGCTTTTCATGACAATGAAGAGACATTTTTGAAAAAATACTTATATGAAATTGCCAGAAGACATCCTTACTTTTATGCCCCGGAACTCCTTTTCTTTGCTAAAAGGTATAAAGCTGCTTTTACAGAATGTTGCCAAGCTGCTGATAAAGCTGCCTGCCTGTTGCCAAAGCTCGATGAACTTCGGGATGAAGGGAAGGCTTCGTCTGCCAAACAGAGACTCAAGTGTGCCAGTCTCCAAAAATTTGGAGAAAGAGCTTTCAAAGCATGGGCAGTAGCTCGCCTGAGCCAGAGATTTCCCAAAGCTGAGTTTGCAGAAGTTTCCAAGTTAGTGACAGATCTTACCAAAGTCCACACGGAATGCTGCCATGGAGATCTGCTTGAATGTGCTGATGACAGGGCGGACCTTGCCAAGTATATCTGTGAAAATCAAGATTCGATCTCCAGTAAACTGAAGGAATGCTGTGAAAAACCTCTGTTGGAAAAATCCCACTGCATTGCCGAAGTGGAAAATGATGAGATGCCTGCTGACTTGCCTTCATTAGCTGCTGATTTTGTTGAAAGTAAGGATGTTTGCAAAAACTATGCTGAGGCAAAGGATGTCTTCCTGGGCATGTTTTTGTATGAATATGCAAGAAGGCATCCTGATTACTCTGTCGTGCTGCTGCTGAGACTTGCCAAGACATATGAAACCACTCTAGAGAAGTGCTGTGCCGCTGCAGATCCTCATGAATGCTATGCCAAAGTGTTCGATGAATTTAAACCTCTTGTGGAAGAGCCTCAGAATTTAATCAAACAAAATTGTGAGCTTTTTGAGCAGCTTGGAGAGTACAAATTCCAGAATGCGCTATTAGTTCGTTACACCAAGAAAGTACCCCAAGTGTCAACTCCAACTCTTGTAGAGGTCTCAAGAAACCTAGGAAAAGTGGGCAGCAAATGTTGTAAACATCCTGAAGCAAAAAGAATGCCCTGTGCAGAAGACTATCTATCCGTGGTCCTGAACCAGTTATGTGTGTTGCATGAGAAAACGCCAGTAAGTGACAGAGTCACCAAATGCTGCACAGAATCCTTGGTGAACAGGCGACCATGCTTTTCAGCTCTGGAAGTCGATGAAACATACGTTCCCAAAGAGTTTAATGCTGAAACATTCACCTTCCATGCAGATATATGCACACTTTCTGAGAAGGAGAGACAAATCAAGAAACAAACTGCACTTGTTGAGCTCGTGAAACACAAGCCCAAGGCAACAAAAGAGCAACTGAAAGCTGTTATGGATGATTTCGCAGCTTTTGTAGAGAAGTGCTGCAAGGCTGACGATAAGGAGACCTGCTTTGCCGAGGAGGGTAAAAAACTTGTTGCTGCAAGTCAAGCTGCCTTAGGCTTA"
print(f"翻译的核苷酸序列: {translate_cds_to_aa(HSA)}")

翻译的核苷酸序列: DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL


In [6]:
arch1 = "GACGCACACAAAAGCGAAGTAGCCCACAGATTTAAAGATTTGGGGGAGGAAAACTTCAAGGCCCTGGTACTCATAGCATTTGCCCAGTATCTGCAGCAATGTCCGTTTGAAGACCATGTCAAGTTGGTCAATGAGGTCACCGAGTTTGCTAAAACTTGCGTTGCTGATGAGAGCGCTGAAAACTGCGATAAGTCTCTCCATACTCTCTTTGGAGACAAGCTCTGTACAGTCGCAACCTTGAGAGAGACTTACGGTGAAATGGCTGATTGCTGTGCCAAACAGGAGCCTGAAAGAAATGAGTGCTTTTTGCAACACAAGGATGACAATCCTAATCTTCCGAGACTTGTCAGACCGGAGGTCGACGTCATGTGTACTGCTTTCCATGACAATGAAGAGACATTTTTGAAAAAATATCTGTATGAAATTGCCCGTAGGCACCCATATTTTTATGCTCCAGAATTGCTGTTCTTTGCAAAAAGATATAAGGCAGCCTTTACCGAGTGTTGTCAGGCCGCCGATAAGGCGGCATGTCTGTTGCCAAAACTAGATGAGCTTAGAGATGAAGGAAAGGCTTCGAGTGCTAAACAGAGATTGAAATGCGCCTCGTTACAAAAATTTGGTGAGAGAGCTTTCAAGGCTTGGGCAGTTGCACGGCTATCCCAGAGGTTTCCTAAAGCTGAGTTTGCGGAAGTTTCCAAGCTGGTCACTGATCTTACCAAAGTTCATACAGAGTGTTGTCATGGAGACCTCCTAGAATGTGCTGATGATCGTGCTGATCTGGCAAAGTATATCTGTGAGAATCAAGACTCTATTTCATCCAAACTGAAGGAATGTTGTGAGAAGCCATTATTGGAAAAGAGTCACTGTATAGCTGAAGTCGAAAACGACGAAATGCCAGCTGATCTTCCCTCTTTAGCAGCCGACTTTGTAGAGTCCAAAGACGTTTGTAAGAATTATGCTGAAGCCAAGGATGTATTTTTGGGAATGTTCTTGTATGAGTATGCTAGACGTCACCCTGACTATTCAGTTGTTTTGTTGTTAAGATTAGCGAAGACTTACGAAACTACTCTTGAGAAATGTTGTGCTGCAGCAGATCCCCACGAATGTTATGCCAAGGTATTTGATGAATTCAAGCCTCTTGTTGAGGAACCCCAAAATTTGATCAAGCAGAACTGTGAGCTTTTTGAACAGCTAGGTGAGTATAAGTTCCAAAATGCCCTATTAGTCAGATATACAAAGAAGGTTCCTCAGGTGTCTACTCCCACTCTGGTCGAGGTTTCCAGAAATCTAGGAAAGGTAGGATCCAAGTGTTGCAAACATCCAGAAGCTAAACGGATGCCGTGTGCAGAAGATTATTTGAGTGTTGTCCTGAACCAATTGTGTGTTCTGCATGAGAAGACGCCAGTAAGTGATCGCGTAACTAAGTGTTGCACCGAATCGTTAGTTAATAGAAGACCTTGCTTCAGTGCCCTAGAAGTTGATGAAACTTATGTTCCCAAGGAGTTTAACGCTGAAACTTTTACGTTCCATGCTGATATTTGCACATTGAGCGAGAAGGAGCGACAAATTAAGAAGCAAACGGCTTTGGTAGAATTGGTTAAACACAAACCTAAGGCAACTAAGGAGCAATTGAAGGCAGTCATGGACGACTTTGCTGCATTCGTTGAAAAGTGTTGTAAAGCTGATGATAAAGAAACTTGTTTTGCTGAAGAAGGAAAGAAATTAGTAGCAGCATCTCAAGCAGCATTGGGATTA"
print(f"翻译的核苷酸序列: {translate_cds_to_aa(arch1)}")

翻译的核苷酸序列: DAHKSEVAHRFKDLGEENFKALVLIAFAQYLQQCPFEDHVKLVNEVTEFAKTCVADESAENCDKSLHTLFGDKLCTVATLRETYGEMADCCAKQEPERNECFLQHKDDNPNLPRLVRPEVDVMCTAFHDNEETFLKKYLYEIARRHPYFYAPELLFFAKRYKAAFTECCQAADKAACLLPKLDELRDEGKASSAKQRLKCASLQKFGERAFKAWAVARLSQRFPKAEFAEVSKLVTDLTKVHTECCHGDLLECADDRADLAKYICENQDSISSKLKECCEKPLLEKSHCIAEVENDEMPADLPSLAADFVESKDVCKNYAEAKDVFLGMFLYEYARRHPDYSVVLLLRLAKTYETTLEKCCAAADPHECYAKVFDEFKPLVEEPQNLIKQNCELFEQLGEYKFQNALLVRYTKKVPQVSTPTLVEVSRNLGKVGSKCCKHPEAKRMPCAEDYLSVVLNQLCVLHEKTPVSDRVTKCCTESLVNRRPCFSALEVDETYVPKEFNAETFTFHADICTLSEKERQIKKQTALVELVKHKPKATKEQLKAVMDDFAAFVEKCCKADDKETCFAEEGKKLVAASQAALGL


In [7]:
!pip install biopython

In [12]:
from Bio.Align import PairwiseAligner
from Bio.Seq import Seq

# Define sequences
# seq1 = Seq("GACAGCTAGCA")
# seq2 = Seq("GACAGCTAG")

predicted_cds_string = "GACGCTCACAAATCAGAAGTAGCTCATCGTTTTAAAGACTTGGGCGAAGAAAATTTCAAGGCTTTAGTTTTAATAGCTTTTGCCCAATATCTTCAACAATGTCCATTTGAAGATCATGTCAAATTGGTCAATGAAGTGACTGAGTTTGCCAAAACTTGTGTTGCTGATGAGTCAGCTGAAAACTGTGATAAATCATTGCATACTTTGTTTGGTGATAAACTTTGTACTGTTGCTACATTGCGTGAAACTTACGGAGAGATGGCAGACTGTTGTGCAAAGCAAGAACCAGAGCGAAATGAGTGCTTTTTACAACATAAAGATGATAATCCGAATTTACCCCGGCTAGTACGTCCAGAAGTGGATGTTATGTGTACTGCGTTCCATGATAATGAGGAGACATTTCTGAAAAAGTACTTGTATGAGATTGCAAGGAGGCACCCATATTTCTATGCTCCAGAGTTACTTTTTTTTGCCAAAAGGTACAAGGCTGCCTTCACCGAATGCTGCCAAGCTGCTGATAAAGCGGCTTGTTTGCTTCCTAAATTAGACGAACTCAGAGATGAAGGTAAGGCCTCCAGTGCTAAACAAAGGCTCAAGTGTGCCTCGTTGCAGAAGTTTGGTGAAAGGGCTTTCAAGGCTTGGGCCGTCGCTCGTTTGTCTCAAAGATTTCCCAAAGCCGAGTTTGCCGAAGTTTCTAAACTGGTGACTGACCTTACTAAAGTACATACCGAGTGTTGTCATGGAGATTTGTTGGAGTGTGCTGATGATAGGGCCGACTTGGCCAAGTATATCTGTGAAAACCAGGATTCAATTTCCTCCAAGTTAAAGGAATGTTGTGAGAAACCGCTATTAGAGAAGTCTCACTGTATTGCCGAAGTGGAAAATGACGAAATGCCTGCTGACTTGCCATCGTTGGCTGCGGATTTTGTCGAATCTAAAGATGTTTGTAAGAACTATGCAGAGGCAAAAGATGTCTTTTTGGGTATGTTTCTATATGAGTATGCTAGAAGACACCCAGATTATTCTGTTGTTCTCTTGTTGAGACTTGCTAAAACCTATGAGACCACTCTGGAGAAATGCTGTGCAGCTGCCGATCCTCACGAGTGCTACGCCAAAGTGTTTGATGAATTCAAACCCTTAGTAGAGGAGCCACAAAATCTGATTAAACAAAACTGCGAGCTTTTTGAACAGCTTGGAGAATACAAGTTCCAGAATGCTTTGCTGGTTCGTTACACTAAAAAAGTCCCTCAGGTTTCAACTCCTACGTTAGTTGAAGTATCAAGGAACCTAGGGAAAGTTGGATCTAAATGTTGTAAGCACCCAGAAGCAAAGAGAATGCCATGTGCTGAAGACTATTTAAGTGTTGTTCTCAACCAACTATGTGTTTTGCACGAAAAAACTCCTGTCTCAGACAGAGTGACCAAATGTTGCACCGAATCTCTAGTCAACAGAAGACCTTGTTTTTCAGCACTAGAAGTTGATGAGACTTATGTGCCTAAGGAATTTAACGCTGAGACTTTTACATTCCATGCTGATATCTGCACTTTGTCTGAAAAGGAGAGACAGATAAAAAAACAGACTGCCTTGGTTGAATTGGTCAAACATAAGCCAAAGGCTACCAAAGAGCAACTGAAGGCTGTTATGGATGATTTTGCTGCGTTTGTTGAAAAATGTTGTAAAGCTGACGATAAGGAGACTTGTTTTGCTGAAGAAGGCAAGAAGCTGGTCGCAGCTTCGCAAGCAGCTTTAGGGCTC"
# Initialize Aligner
aligner = PairwiseAligner()
aligner.mode = 'global' # or 'local'
aligner.match_score = 1
aligner.mismatch_score = -1

# Perform Alignment
alignments = aligner.align(predicted_cds_string, arch1)

# Print the top alignment
print(alignments[0])


target            0 GACGCT-CACAAATCAG--AAGTAGCTC-ATC-G-TTTTAAAGACTT-GGGCG-AAG-AA
                  0 |||||--||||||--||--|||||||-|-|-|-|-|||-|||||-||-|||-|-|-|-||
query             0 GACGC-ACACAAA--AGCGAAGTAGC-CCA-CAGATTT-AAAGA-TTTGGG-GGA-GGAA

target           51 AA-TTTCAAGGC--T--TTAG-TTTT-AATAGC-TTTTGCCCAA-TATCTT-CAA-CAAT
                 60 ||-||-||||||--|--|-|--|----|-||||-|||-|||||--|||||--||--||||
query            51 AACTT-CAAGGCCCTGGT-A-CT---CA-TAGCATTT-GCCCA-GTATCT-GCA-GCAAT

target          100 GTCCA-TTTGAAGATC-ATGTCAAA-TTGGTCAATGAAGTG--ACT-GAGTTTGCC-AAA
                120 ||||--||||||||-|-|||||||--|||||||||||-|-|--||--||||||||--|||
query           100 GTCC-GTTTGAAGA-CCATGTCAA-GTTGGTCAATGA-G-GTCAC-CGAGTTTGC-TAAA

target          153 ACTTGT-GTTGCTGATGAGTCAGC--TGAAAACTGT-GATAAA-TCAT-TGC-ATACT-T
                180 |||||--||||||||||||--|||--|||||||||--|||||--||-|-|-|-|||||-|
query           153 ACTTG-CGTTGCTGATGAG--AGCGCTGAAAACTG-CGATAA-GTC-TCT-CCATACTCT

target          205 -TGT